# ✈️ DelayPredict — 03b_v2 XGBoost (Feature Engineering Experiments)

Extends `03b_xgboost.ipynb` with three toggleable feature engineering strategies.

| Toggle | Strategy |
|---|---|
| `USE_TARGET_ENC_COMBINATIONS` | Pairwise interactions: Airline×Hour, Route×Day |
| `USE_HASH_VECTORIZER` | Hash-bucket Route into 64 numeric columns |
| `USE_STRING_COMBINATIONS` | 3-way string: Airline_Route_Hour |

---
**Input:** `data/raw/airlines_delay.csv`  
**Input (optional):** `data/processed/baseline_metrics.csv`, `data/processed/rf_metrics.csv`, `data/processed/xgb_metrics.csv`  
**Output:** `models/xgb_v2_model.pkl`, `data/processed/xgb_v2_metrics.csv`

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder
from sklearn.feature_extraction import FeatureHasher
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
    ConfusionMatrixDisplay, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay,
)
from xgboost import XGBClassifier
import joblib
import mlflow
import mlflow.sklearn
mlflow.set_tracking_uri("http://127.0.0.1:5000") #Muss in alle Notebooks, damit die Experimente im MLflow-UI angezeigt werden

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42


## 2. Feature Engineering Toggles

Set each flag to `True` or `False`, then re-run the notebook top to bottom.

In [ ]:
# ── Toggle each strategy independently ──────────────────────────────────────
USE_TARGET_ENC_COMBINATIONS = True   # Airline×Hour, Route×DayOfWeek, Airline×Route
USE_HASH_VECTORIZER         = True   # Route → 64 hash bucket columns
USE_STRING_COMBINATIONS     = True   # Airline_Route_Hour → 1 target-encoded column

print(f'USE_TARGET_ENC_COMBINATIONS : {USE_TARGET_ENC_COMBINATIONS}')
print(f'USE_HASH_VECTORIZER         : {USE_HASH_VECTORIZER}')
print(f'USE_STRING_COMBINATIONS     : {USE_STRING_COMBINATIONS}')


## 3. Load Raw Data

Always loads from raw — fully self-contained.

In [ ]:
RAW_PATH = Path('../data/raw/airlines_delay.csv')

if not RAW_PATH.exists():
    raise FileNotFoundError(f'Dataset not found at {RAW_PATH}')

df = pd.read_csv(RAW_PATH)
print('Source :', RAW_PATH)
print('Shape  :', df.shape)
display(df.head())


## 4. Base Feature Engineering

Core transformations applied in every run — independent of toggles.

| Step | Action |
|---|---|
| `Time` → `DepartureHour` | `Time // 60` — minutes since midnight |
| `Route` | `AirportFrom + '-' + AirportTo` |

In [ ]:
# Time is minutes since midnight (0=00:00, 1439=23:59)
df['DepartureHour'] = df['Time'] // 60

# Route captures origin-destination delay pattern
df['Route'] = df['AirportFrom'] + '-' + df['AirportTo']

df = df.drop(columns=['id', 'Flight', 'Time'])

print('Base columns:', df.columns.tolist())


## 5. Experimental Feature Engineering

Each block runs only if its toggle is enabled.

In [ ]:
# ── Idea 1: Pairwise target-encoding combinations ────────────────────────────
# Each string encodes a 2-way interaction as its historical delay rate.
if USE_TARGET_ENC_COMBINATIONS:
    df['Airline_Hour']    = df['Airline'].astype(str) + '_' + df['DepartureHour'].astype(str)
    df['Route_DayOfWeek'] = df['Route'].astype(str)   + '_' + df['DayOfWeek'].astype(str)
    df['Airline_Route']   = df['Airline'].astype(str) + '_' + df['Route'].astype(str)
    print('[Idea 1] Added: Airline_Hour, Route_DayOfWeek, Airline_Route')
else:
    print('[Idea 1] Skipped')

# ── Idea 2: Hash Vectorizer for Route ────────────────────────────────────────
# Maps ~2000 routes into 64 hash buckets — memory efficient but lossy.
if USE_HASH_VECTORIZER:
    hasher = FeatureHasher(n_features=64, input_type='string')
    route_hashed = hasher.transform(df['Route'].apply(lambda x: [x]))
    hash_cols = [f'RouteHash_{i}' for i in range(64)]
    df_hashed = pd.DataFrame(route_hashed.toarray(), columns=hash_cols, index=df.index)
    df = pd.concat([df, df_hashed], axis=1)
    print(f'[Idea 2] Added: {len(hash_cols)} RouteHash columns')
else:
    print('[Idea 2] Skipped')

# ── Idea 3: 3-way string combination ─────────────────────────────────────────
# Combines Airline + Route + DepartureHour into one string — captures 3-way interaction.
if USE_STRING_COMBINATIONS:
    df['Airline_Route_Hour'] = (
        df['Airline'].astype(str) + '_' +
        df['Route'].astype(str)   + '_' +
        df['DepartureHour'].astype(str)
    )
    print(f'[Idea 3] Added: Airline_Route_Hour ({df["Airline_Route_Hour"].nunique():,} unique values)')
else:
    print('[Idea 3] Skipped')

print(f'\nTotal columns: {len(df.columns)}')


## 6. Features and Target

Feature lists built dynamically from active toggles.

In [ ]:
TARGET = 'Delay'

# ── Base features (always used) ──────────────────────────────────────────────
BASE_TARGET_ENC = ['Airline', 'AirportFrom', 'AirportTo', 'Route']
NUMERIC         = ['Length', 'DepartureHour', 'DayOfWeek']

# ── Toggle features ───────────────────────────────────────────────────────────
EXTRA_TARGET_ENC = []
HASH_COLS        = []

if USE_TARGET_ENC_COMBINATIONS:
    EXTRA_TARGET_ENC += ['Airline_Hour', 'Route_DayOfWeek', 'Airline_Route']

if USE_HASH_VECTORIZER:
    HASH_COLS = [f'RouteHash_{i}' for i in range(64)]

if USE_STRING_COMBINATIONS:
    EXTRA_TARGET_ENC += ['Airline_Route_Hour']

# ── Final feature lists ───────────────────────────────────────────────────────
ALL_TARGET_ENC = BASE_TARGET_ENC + EXTRA_TARGET_ENC
ALL_FEATURES   = NUMERIC + ALL_TARGET_ENC + HASH_COLS

X = df[ALL_FEATURES]
y = df[TARGET]

print(f'Numeric        : {NUMERIC}')
print(f'Target-encoded : {ALL_TARGET_ENC}')
print(f'Hash columns   : {len(HASH_COLS)}')
print(f'Total features : {len(ALL_FEATURES)}')
print(f'X shape        : {X.shape}')

dist = y.value_counts(normalize=True).mul(100).round(1).rename('% share')
print('\nTarget distribution:')
display(dist)


## 7. Train / Test Split

80/20 split with stratify — identical to all other notebooks for fair comparison.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f'Train: {X_train.shape}  |  delay rate: {y_train.mean():.3f}')
print(f'Test : {X_test.shape}   |  delay rate: {y_test.mean():.3f}')


## 8. Preprocessing Pipeline

Combines numeric features (passthrough) and target-encoded categoricals
into a single feature matrix for the model.

- `TargetEncoder` replaces each category with its smoothed historical delay rate
- `smooth='auto'` prevents overfitting on rare categories
- Fitted on training data only — no leakage into test set

In [ ]:
# Build transformer list dynamically
transformers = [
    ('num',        'passthrough',                                        NUMERIC),
    ('target_enc', TargetEncoder(smooth='auto', random_state=RANDOM_STATE), ALL_TARGET_ENC),
]

# Hash columns are already numeric — just pass through
if HASH_COLS:
    transformers.append(('hash_pass', 'passthrough', HASH_COLS))

preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

print('Active transformers:')
for name, _, cols in transformers:
    print(f'  {name:<15}: {len(cols)} features')


## 9. XGBoost Model

| Parameter | Value | Reason |
|---|---|---|
| `n_estimators` | 500 | More boosting rounds |
| `learning_rate` | 0.05 | Small steps — precise with 500 rounds |
| `max_depth` | 6 | Standard depth |
| `subsample` | 0.8 | Row sampling — reduces overfitting |
| `colsample_bytree` | 0.8 | Feature sampling per tree |
| `min_child_weight` | 10 | Minimum samples per leaf — regularization |

In [ ]:
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=10,
        eval_metric='logloss',
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])

xgb_pipeline.fit(X_train, y_train)
print('XGBoost trained successfully.')


## 10. Evaluate

Evaluate on the held-out test set.

| Metric | What it measures |
|---|---|
| Accuracy | Share of correct predictions overall |
| Precision | Of predicted delays, how many were real delays |
| Recall | Of real delays, how many did we catch |
| F1 | Harmonic mean of Precision and Recall |
| ROC-AUC | Ability to rank delayed flights above non-delayed ones |

In [ ]:
y_pred_xgb  = xgb_pipeline.predict(X_test)
y_proba_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]

xgb_v2_metrics = {
    'Accuracy' : accuracy_score(y_test, y_pred_xgb),
    'Precision': precision_score(y_test, y_pred_xgb, zero_division=0),
    'Recall'   : recall_score(y_test, y_pred_xgb, zero_division=0),
    'F1'       : f1_score(y_test, y_pred_xgb, zero_division=0),
    'ROC-AUC'  : roc_auc_score(y_test, y_proba_xgb),
}

print('XGBoost v2 metrics:')
for k, v in xgb_v2_metrics.items():
    print(f'  {k:<10}: {v:.4f}')

print()
print(classification_report(y_test, y_pred_xgb,
      target_names=['No Delay', 'Delay'], zero_division=0))


## 11. Comparison Against All Models

In [ ]:
BASELINE_PATH = Path('../data/processed/baseline_metrics.csv')
RF_PATH       = Path('../data/processed/rf_metrics.csv')
XGB_PATH      = Path('../data/processed/xgb_metrics.csv')

xgb_v2_row = {'model': 'XGBoost_v2', **xgb_v2_metrics}

prior = []
for p in [BASELINE_PATH, RF_PATH, XGB_PATH]:
    if p.exists():
        prior.append(pd.read_csv(p))
    else:
        print(f'Not found: {p}')

comparison_df = pd.concat(prior + [pd.DataFrame([xgb_v2_row])], ignore_index=True)
display(comparison_df.round(4))


## 12. Confusion Matrix

- **False Negatives (bottom-left):** predicted No Delay, actually Delay ← the costly error
- **True Positives (bottom-right):** correctly predicted Delay

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_xgb,
    display_labels=['No Delay', 'Delay'],
    cmap='Blues', ax=ax,
)
ax.set_title('Confusion Matrix — XGBoost v2')
plt.tight_layout()
plt.show()


## 13. ROC Curve and Precision-Recall Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

RocCurveDisplay.from_predictions(y_test, y_proba_xgb, ax=ax1, name='XGBoost v2')
ax1.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.5)')
ax1.set_title('ROC Curve — XGBoost v2')
ax1.legend()

PrecisionRecallDisplay.from_predictions(y_test, y_proba_xgb, ax=ax2, name='XGBoost v2')
ax2.axhline(y_test.mean(), color='k', linestyle='--',
            label=f'No-skill ({y_test.mean():.2f})')
ax2.set_title('Precision-Recall Curve — XGBoost v2')
ax2.legend()

plt.tight_layout()
plt.show()


## 14. Feature Importance

Interaction features (Idea 1 & 3) should rank higher than base features if they add signal.

In [ ]:
# Feature names match ColumnTransformer output order: numeric → target_enc → hash
feature_names = NUMERIC + ALL_TARGET_ENC + HASH_COLS
importances   = xgb_pipeline.named_steps['classifier'].feature_importances_

importance_df = (
    pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)

print('Top 20 most important features:')
display(importance_df.head(20))

# Bar chart: top 15 non-hash features
plot_df = (
    importance_df[~importance_df['Feature'].str.startswith('RouteHash_')]
    .head(15)
    .sort_values('Importance')
)

plt.figure(figsize=(9, 5))
plt.barh(plot_df['Feature'], plot_df['Importance'], color='steelblue')
plt.title('Top 15 Feature Importances — XGBoost v2')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()


## 15. Predicted Probability Distribution

Better separation = better model confidence. Compare to `03b_xgboost.ipynb`.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(y_proba_xgb[y_test == 0], bins=50, alpha=0.6,
         label='No Delay (actual)', color='steelblue')
plt.hist(y_proba_xgb[y_test == 1], bins=50, alpha=0.6,
         label='Delay (actual)', color='tomato')
plt.axvline(0.5, color='black', linestyle='--', label='Threshold 0.5')
plt.title('Predicted Probability Distribution — XGBoost v2')
plt.xlabel('P(Delay)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()


## 16. Save Model and Metrics

In [ ]:
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / 'xgb_v2_model.pkl'
joblib.dump(xgb_pipeline, MODEL_PATH)
print(f'Model saved   : {MODEL_PATH}')

OUTPUT_DIR = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

xgb_v2_metrics_df = pd.DataFrame([{'model': 'XGBoost_v2', **xgb_v2_metrics}])
XGB_V2_PATH = OUTPUT_DIR / 'xgb_v2_metrics.csv'
xgb_v2_metrics_df.to_csv(XGB_V2_PATH, index=False)
print(f'Metrics saved : {XGB_V2_PATH}')


## 17. MLflow Logging

Logs toggles as parameters so every run is reproducible in the UI.  
Start UI: `mlflow ui` → `http://localhost:5000`

In [ ]:
mlflow.set_experiment('DelayPredict')

with mlflow.start_run(run_name='XGBoost_v2_FeatureExperiment'):
    xgb_clf = xgb_pipeline.named_steps['classifier']

    mlflow.log_params({
        'model'                      : 'XGBoost_v2',
        'USE_TARGET_ENC_COMBINATIONS': USE_TARGET_ENC_COMBINATIONS,
        'USE_HASH_VECTORIZER'        : USE_HASH_VECTORIZER,
        'USE_STRING_COMBINATIONS'    : USE_STRING_COMBINATIONS,
        'n_estimators'               : xgb_clf.n_estimators,
        'learning_rate'              : xgb_clf.learning_rate,
        'max_depth'                  : xgb_clf.max_depth,
        'total_features'             : len(ALL_FEATURES),
        'random_state'               : RANDOM_STATE,
    })

    mlflow.log_metrics({
        'accuracy' : xgb_v2_metrics['Accuracy'],
        'precision': xgb_v2_metrics['Precision'],
        'recall'   : xgb_v2_metrics['Recall'],
        'f1'       : xgb_v2_metrics['F1'],
        'roc_auc'  : xgb_v2_metrics['ROC-AUC'],
    })

    mlflow.sklearn.log_model(xgb_pipeline, artifact_path='model')

    print('MLflow run logged — experiment: DelayPredict')
    print(f'Toggles: TEC={USE_TARGET_ENC_COMBINATIONS}, HV={USE_HASH_VECTORIZER}, SC={USE_STRING_COMBINATIONS}')


## 18. Summary

In [ ]:
print('XGBOOST v2 RESULTS')
print('=' * 72)
print(f'Train : {X_train.shape[0]:>7,} rows')
print(f'Test  : {X_test.shape[0]:>7,} rows')
print(f'Features ({len(ALL_FEATURES)}): {ALL_FEATURES}')
print()
print(f'  Target Enc Combinations : {USE_TARGET_ENC_COMBINATIONS}')
print(f'  Hash Vectorizer         : {USE_HASH_VECTORIZER}')
print(f'  String Combinations     : {USE_STRING_COMBINATIONS}')
print()

metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
print(f"{'Model':<22}" + ''.join(f'{m:>10}' for m in metric_cols))
print('-' * 72)
for _, row in comparison_df.iterrows():
    vals = ''.join(f"{row[m]:>10.4f}" for m in metric_cols)
    print(f"{row['model']:<22}{vals}")
